In [ ]:
# =============================================================================
# 셀 1: 환경 설정
# =============================================================================

import subprocess, os, sys

subprocess.run(["pip", "install", "-q",
                "scikit-learn", "matplotlib", "seaborn",
                "pandas", "numpy"], check=False)

# 공통 라이브러리 import
import torch
import torch.nn as nn
import torch.nn.functional as F
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import (roc_auc_score, balanced_accuracy_score,
                              f1_score, confusion_matrix)

# 경로 설정 (TNBC_PROJECT_ROOT 환경변수로 변경 가능, 기본값: ~/TCGA_BRCA_project)
PROJECT_ROOT        = os.environ.get('TNBC_PROJECT_ROOT', os.path.expanduser('~/TCGA_BRCA_project'))
FEATURE_DIR = f'{PROJECT_ROOT}/features'
LOG_DIR     = f'{PROJECT_ROOT}/logs'
RESULT_DIR  = f'{PROJECT_ROOT}/results'
os.makedirs(RESULT_DIR, exist_ok=True)

BEST_MODEL_PATH = f'{RESULT_DIR}/best_model.pt'
CLAM_CSV_PATH   = f'{LOG_DIR}/dataset_for_clam.csv'

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print("✓ 환경 설정 완료")
print(f"  디바이스       : {device}")
print(f"  특징 파일 경로 : {FEATURE_DIR}")
print(f"  결과 저장 경로 : {RESULT_DIR}")

In [ ]:
# =============================================================================
# 셀 2: 데이터셋 로드 및 분할
# =============================================================================

dataset_df = pd.read_csv(CLAM_CSV_PATH)

# file_path 컬럼을 현재 로컬 경로 기준으로 재생성
# (저장 시 경로가 고정되므로 항상 일치)
dataset_df['file_path'] = dataset_df['slide_id'].apply(
    lambda sid: os.path.join(FEATURE_DIR, f'{sid}.pt')
)

print(f"전체 데이터셋: {len(dataset_df)}명")
print(f"라벨 분포:\n{dataset_df['label_str'].value_counts().to_string()}")

# 70 / 15 / 15 층화 분할 (random_state=42 고정 → 재현성 보장)
train_df, temp_df = train_test_split(
    dataset_df, test_size=0.30,
    stratify=dataset_df['label'], random_state=42
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.50,
    stratify=temp_df['label'], random_state=42
)

print(f"\n데이터셋 분할 결과 (70 / 15 / 15):")
for name, df in [("Train", train_df), ("Val", val_df), ("Test", test_df)]:
    print(f"  {name:5s}: {len(df)}명  "
          f"TNBC {(df['label']==1).sum()} / "
          f"non-TNBC {(df['label']==0).sum()}")

# 분할 결과 저장 (셀 7 단독 실행 시 재사용)
train_df.to_csv(f'{LOG_DIR}/split_train.csv', index=False)
val_df.to_csv(  f'{LOG_DIR}/split_val.csv',   index=False)
test_df.to_csv( f'{LOG_DIR}/split_test.csv',  index=False)
print("\n✓ 분할 CSV 저장 완료")



In [ ]:
# =============================================================================
# 셀 3: Dataset 및 DataLoader 정의
# =============================================================================

class BagDataset(Dataset):
    """
    CLAM 입력용 Dataset.
    패치 수가 슬라이드마다 다르므로 batch_size=1 고정.
    """
    def __init__(self, df):
        self.df = df.reset_index(drop=True)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row      = self.df.iloc[idx]
        data     = torch.load(row['file_path'], map_location='cpu',
                              weights_only=False)  # PyTorch 2.x 호환
        features = data['features']   # (n_patches, 1024)
        label    = torch.tensor(data['label'], dtype=torch.long)
        slide_id = data['patient_id']
        return features, label, slide_id

def make_loader(df, shuffle=False):
    return DataLoader(BagDataset(df), batch_size=1,
                      shuffle=shuffle, num_workers=0)

train_loader = make_loader(train_df, shuffle=True)
val_loader   = make_loader(val_df,   shuffle=False)
test_loader  = make_loader(test_df,  shuffle=False)

print(f"✓ DataLoader 생성 완료")
print(f"  Train: {len(train_loader)}개 / Val: {len(val_loader)}개 / Test: {len(test_loader)}개")

In [ ]:
# =============================================================================
# 셀 4: CLAM 모델 정의
# =============================================================================

class Attn_Net_Gated(nn.Module):
    """
    Gated Attention Network.
    tanh 브랜치 × sigmoid 브랜치 → 패치별 attention 가중치 계산.
    """
    def __init__(self, L=512, D=256, dropout=True, n_classes=1):
        # L=512: fc 레이어 거친 후의 hidden_dim
        super().__init__()
        self.attention_a = nn.Sequential(nn.Linear(L, D), nn.Tanh())
        self.attention_b = nn.Sequential(nn.Linear(L, D), nn.Sigmoid())
        self.attention_c = nn.Linear(D, n_classes)
        self.drop = nn.Dropout(0.25) if dropout else nn.Identity()

    def forward(self, x):
        a = self.attention_a(x)
        b = self.attention_b(x)
        A = self.attention_c(self.drop(a * b))  # (n_patches, 1)
        return A, x

class CLAM_SB(nn.Module):
    """
    CLAM Single Branch — 이진 분류.

    흐름:
      (n_patches, 1024) → fc → (n_patches, 512)
      → Gated Attention → attention 가중치 (n_patches, 1)
      → Attention Pooling → 슬라이드 임베딩 (1, 512)
      → classifier → (1, 2)
    """
    def __init__(self, gate=True, dropout=True,
                 k_sample=8, n_classes=2,
                 feat_dim=1024, hidden_dim=512):
        super().__init__()
        self.fc = nn.Sequential(
            nn.Linear(feat_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.25) if dropout else nn.Identity()
        )
        self.attention_net = Attn_Net_Gated(
            L=hidden_dim, D=256, dropout=dropout, n_classes=1
        )
        self.classifier          = nn.Linear(hidden_dim, n_classes)
        self.instance_classifier = nn.Linear(hidden_dim, 2)
        self.k_sample = k_sample

    def forward(self, h, return_attn=False):
        h      = self.fc(h)                          # (n_patches, 512)
        A, h   = self.attention_net(h)               # A: (n_patches, 1)
        A      = F.softmax(A.transpose(0, 1), dim=1) # (1, n_patches)
        M      = torch.mm(A, h)                       # (1, 512)
        logits = self.classifier(M)
        probs  = F.softmax(logits, dim=1)
        pred   = torch.argmax(probs, dim=1)
        if return_attn:
            return logits, probs, pred, A
        return logits, probs, pred

    def get_instance_loss(self, h, label, device):
        """
        CLAM 보조 손실: attention 기반 top-k/bottom-k pseudo-label 생성.
        슬라이드 레이블만으로 인스턴스 단위 학습 신호를 만드는 핵심 메커니즘.
        """
        h_fc = self.fc(h)
        A, _ = self.attention_net(h_fc)
        A    = A.squeeze(1)   # (n_patches,)
        k    = min(self.k_sample, len(A) // 2)

        top_idx    = torch.topk(A, k).indices
        bottom_idx = torch.topk(A, k, largest=False).indices
        top_feats    = h_fc[top_idx]
        bottom_feats = h_fc[bottom_idx]

        # TNBC(1): top-k → 양성, bottom-k → 음성
        # non-TNBC(0): top-k → 음성, bottom-k → 양성
        if label.item() == 1:
            top_lbl    = torch.ones(k,  dtype=torch.long, device=device)
            bottom_lbl = torch.zeros(k, dtype=torch.long, device=device)
        else:
            top_lbl    = torch.zeros(k, dtype=torch.long, device=device)
            bottom_lbl = torch.ones(k,  dtype=torch.long, device=device)

        feats  = torch.cat([top_feats, bottom_feats])
        labels = torch.cat([top_lbl, bottom_lbl])
        return F.cross_entropy(self.instance_classifier(feats), labels)


model = CLAM_SB(gate=True, dropout=True, k_sample=8,
                n_classes=2, feat_dim=1024, hidden_dim=512).to(device)

print(f"✓ CLAM_SB 생성 완료")
print(f"  파라미터 수: {sum(p.numel() for p in model.parameters()):,}개")

In [ ]:
# =============================================================================
# 셀 5: 학습/평가 함수 정의
# =============================================================================

# 클래스 가중치 (TNBC 116 / non-TNBC 214 불균형 보정)
n_tnbc    = (train_df['label'] == 1).sum()
n_nontnbc = (train_df['label'] == 0).sum()
class_weight = torch.tensor(
    [1.0, n_nontnbc / n_tnbc], dtype=torch.float
).to(device)
print(f"클래스 가중치: non-TNBC={class_weight[0]:.2f}, TNBC={class_weight[1]:.2f}")

def train_one_epoch(model, loader, optimizer, bag_weight=0.7):
    model.train()
    total_loss = correct = total = 0

    for features, label, _ in loader:
        features = features.squeeze(0).to(device)
        label    = label.squeeze(0).to(device)
        optimizer.zero_grad()

        logits, _, pred = model(features)
        # weighted cross entropy — TNBC 클래스 불균형 보정
        bag_loss  = F.cross_entropy(logits, label.unsqueeze(0),
                                    weight=class_weight)
        inst_loss = model.get_instance_loss(features, label, device)
        loss      = bag_weight * bag_loss + (1 - bag_weight) * inst_loss

        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        correct    += (pred.item() == label.item())
        total      += 1

    return total_loss / total, correct / total

def evaluate(model, loader):
    model.eval()
    all_labels, all_probs, all_preds = [], [], []

    with torch.no_grad():
        for features, label, _ in loader:
            features = features.squeeze(0).to(device)
            label    = label.squeeze(0).to(device)
            _, probs, pred = model(features)
            all_labels.append(label.item())
            all_probs.append(probs[0, 1].item())  # TNBC(클래스 1) 확률
            all_preds.append(pred.item())

    labels = np.array(all_labels)
    probs  = np.array(all_probs)
    preds  = np.array(all_preds)

    return {
        'auc':     round(roc_auc_score(labels, probs), 4),
        'bal_acc': round(balanced_accuracy_score(labels, preds), 4),
        'f1':      round(f1_score(labels, preds, zero_division=0), 4),
        'cm':      confusion_matrix(labels, preds)
    }


In [ ]:
# =============================================================================
# 셀 6: 학습 실행
# =============================================================================

NUM_EPOCHS   = 30
LR           = 2e-4
WEIGHT_DECAY = 1e-5
BAG_WEIGHT   = 0.7

optimizer = torch.optim.Adam(
    model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY
)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=NUM_EPOCHS, eta_min=1e-6
)

history = {
    'train_loss': [], 'train_acc': [],
    'val_auc':    [], 'val_bal_acc': [], 'val_f1': []
}
best_val_auc = 0.0

print(f"학습 시작 (총 {NUM_EPOCHS} 에폭)")
print("=" * 65)

for epoch in range(1, NUM_EPOCHS + 1):
    train_loss, train_acc = train_one_epoch(
        model, train_loader, optimizer, BAG_WEIGHT
    )
    val_m = evaluate(model, val_loader)
    scheduler.step()

    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_auc'].append(val_m['auc'])
    history['val_bal_acc'].append(val_m['bal_acc'])
    history['val_f1'].append(val_m['f1'])

    flag = ""
    if val_m['auc'] > best_val_auc:
        best_val_auc = val_m['auc']
        torch.save({
            'epoch':       epoch,
            'model_state': model.state_dict(),
            'val_auc':     val_m['auc'],
            'val_bal_acc': val_m['bal_acc'],
            'val_f1':      val_m['f1'],
            # 셀 7 단독 실행을 위한 분할 정보 함께 저장
            'split': {
                'train_ids': train_df['slide_id'].tolist(),
                'val_ids':   val_df['slide_id'].tolist(),
                'test_ids':  test_df['slide_id'].tolist(),
            }
        }, BEST_MODEL_PATH)
        flag = "  ← Best"

    print(f"Epoch {epoch:02d}/{NUM_EPOCHS} | "
          f"Loss {train_loss:.4f}  Acc {train_acc:.3f} | "
          f"Val AUC {val_m['auc']:.4f}  "
          f"BalAcc {val_m['bal_acc']:.4f}  "
          f"F1 {val_m['f1']:.4f}"
          f"{flag}")

print(f"\nBest Val AUC: {best_val_auc:.4f}")

# 학습 곡선 시각화
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].plot(history['train_loss'])
axes[0].set_title('Training Loss'); axes[0].set_xlabel('Epoch')
axes[1].plot(history['val_auc'],     label='Val AUC')
axes[1].plot(history['val_bal_acc'], label='Val Balanced Acc')
axes[1].set_title('Validation Metrics'); axes[1].set_xlabel('Epoch'); axes[1].legend()
axes[2].plot(history['val_f1'], color='green')
axes[2].set_title('Val F1 Score'); axes[2].set_xlabel('Epoch')
plt.tight_layout()
curve_path = f'{RESULT_DIR}/training_curves.png'
plt.savefig(curve_path, dpi=150, bbox_inches='tight')
plt.show()
print(f"학습 곡선 저장: {curve_path}")

In [ ]:
# =============================================================================
# 셀 7: 테스트 평가 + Confusion Matrix
# (세션 재시작 후 셀 1 → 셀 7만 실행해도 독립 동작)
# =============================================================================

# ── 라이브러리/경로/모델 — 셀 7 단독 실행 시를 위해 재선언 ────────────────────
import os, torch, pandas as pd, numpy as np
import matplotlib.pyplot as plt, seaborn as sns
import torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import (roc_auc_score, balanced_accuracy_score,
                              f1_score, confusion_matrix)

PROJECT_ROOT        = os.environ.get('TNBC_PROJECT_ROOT', os.path.expanduser('~/TCGA_BRCA_project'))
FEATURE_DIR = f'{PROJECT_ROOT}/features'
LOG_DIR     = f'{PROJECT_ROOT}/logs'
RESULT_DIR  = f'{PROJECT_ROOT}/results'
BEST_MODEL_PATH   = f'{RESULT_DIR}/best_model.pt'
CLAM_CSV_PATH     = f'{LOG_DIR}/dataset_for_clam.csv'
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# 모델 구조 재정의 (셀 4가 실행되지 않은 경우 대비)
class Attn_Net_Gated(nn.Module):
    def __init__(self, L=512, D=256, dropout=True, n_classes=1):
        super().__init__()
        self.attention_a = nn.Sequential(nn.Linear(L, D), nn.Tanh())
        self.attention_b = nn.Sequential(nn.Linear(L, D), nn.Sigmoid())
        self.attention_c = nn.Linear(D, n_classes)
        self.drop = nn.Dropout(0.25) if dropout else nn.Identity()
    def forward(self, x):
        a = self.attention_a(x)
        b = self.attention_b(x)
        A = self.attention_c(self.drop(a * b))
        return A, x

class CLAM_SB(nn.Module):
    def __init__(self, gate=True, dropout=True,
                 k_sample=8, n_classes=2,
                 feat_dim=1024, hidden_dim=512):
        super().__init__()
        self.fc = nn.Sequential(
            nn.Linear(feat_dim, hidden_dim), nn.ReLU(),
            nn.Dropout(0.25) if dropout else nn.Identity()
        )
        self.attention_net = Attn_Net_Gated(
            L=hidden_dim, D=256, dropout=dropout, n_classes=1
        )
        self.classifier          = nn.Linear(hidden_dim, n_classes)
        self.instance_classifier = nn.Linear(hidden_dim, 2)
        self.k_sample = k_sample
    def forward(self, h, return_attn=False):
        h      = self.fc(h)
        A, h   = self.attention_net(h)
        A      = F.softmax(A.transpose(0, 1), dim=1)
        M      = torch.mm(A, h)
        logits = self.classifier(M)
        probs  = F.softmax(logits, dim=1)
        pred   = torch.argmax(probs, dim=1)
        if return_attn:
            return logits, probs, pred, A
        return logits, probs, pred

class BagDataset(Dataset):
    def __init__(self, df):
        self.df = df.reset_index(drop=True)
    def __len__(self):
        return len(self.df)
    def __getitem__(self, idx):
        row  = self.df.iloc[idx]
        data = torch.load(row['file_path'], map_location='cpu',
                          weights_only=False)
        return (data['features'],
                torch.tensor(data['label'], dtype=torch.long),
                data['patient_id'])

def evaluate(model, loader):
    model.eval()
    all_labels, all_probs, all_preds = [], [], []
    with torch.no_grad():
        for features, label, _ in loader:
            features = features.squeeze(0).to(device)
            label    = label.squeeze(0).to(device)
            _, probs, pred = model(features)
            all_labels.append(label.item())
            all_probs.append(probs[0, 1].item())
            all_preds.append(pred.item())
    labels = np.array(all_labels)
    probs  = np.array(all_probs)
    preds  = np.array(all_preds)
    return {
        'auc':     round(roc_auc_score(labels, probs), 4),
        'bal_acc': round(balanced_accuracy_score(labels, preds), 4),
        'f1':      round(f1_score(labels, preds, zero_division=0), 4),
        'cm':      confusion_matrix(labels, preds)
    }

# ── Best 모델 로드 ─────────────────────────────────────────────────────────────
checkpoint = torch.load(BEST_MODEL_PATH, map_location=device,
                        weights_only=False)
model = CLAM_SB(gate=True, dropout=True, k_sample=8,
                n_classes=2, feat_dim=1024, hidden_dim=512).to(device)
model.load_state_dict(checkpoint['model_state'])
print(f"✓ Best 모델 로드  "
      f"(Epoch {checkpoint['epoch']}, Val AUC {checkpoint['val_auc']:.4f})")

# ── Test 분할 복원 ─────────────────────────────────────────────────────────────
# 방법 1: split_test.csv가 있으면 그대로 로드
test_csv_path = f'{LOG_DIR}/split_test.csv'
if os.path.exists(test_csv_path):
    test_df = pd.read_csv(test_csv_path)
    print(f"✓ split_test.csv 로드  ({len(test_df)}명)")
else:
    # 방법 2: checkpoint 안에 저장된 split 정보로 복원
    print("split_test.csv 없음 → checkpoint의 split 정보로 복원")
    dataset_df = pd.read_csv(CLAM_CSV_PATH)
    test_ids   = checkpoint['split']['test_ids']
    test_df    = dataset_df[dataset_df['slide_id'].isin(test_ids)].reset_index(drop=True)
    print(f"✓ Test 분할 복원  ({len(test_df)}명)")

# file_path 재생성 (로컬 경로 기준)
test_df['file_path'] = test_df['slide_id'].apply(
    lambda sid: os.path.join(FEATURE_DIR, f'{sid}.pt')
)

print(f"  라벨 분포: TNBC {(test_df['label']==1).sum()} / "
      f"non-TNBC {(test_df['label']==0).sum()}")

# ── 테스트 평가 ───────────────────────────────────────────────────────────────
test_loader = DataLoader(BagDataset(test_df), batch_size=1,
                         shuffle=False, num_workers=0)
test_m = evaluate(model, test_loader)

print(f"\n{'='*45}")
print("최종 테스트 성능")
print(f"{'='*45}")
print(f"  AUC               : {test_m['auc']:.4f}")
print(f"  Balanced Accuracy : {test_m['bal_acc']:.4f}")
print(f"  F1 Score          : {test_m['f1']:.4f}")

# ── Confusion Matrix 시각화 ───────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(
    test_m['cm'], annot=True, fmt='d', cmap='Blues',
    xticklabels=['non-TNBC', 'TNBC'],
    yticklabels=['non-TNBC', 'TNBC'], ax=ax
)
ax.set_xlabel('Predicted')
ax.set_ylabel('Actual')
ax.set_title(f'Confusion Matrix  (Test AUC: {test_m["auc"]:.4f})')
plt.tight_layout()
cm_path = f'{RESULT_DIR}/confusion_matrix.png'
plt.savefig(cm_path, dpi=150, bbox_inches='tight')
plt.show()
print(f"✓ Confusion Matrix 저장: {cm_path}")

# ── 결과 요약 저장 ─────────────────────────────────────────────────────────────
summary = {
    'best_epoch':     checkpoint['epoch'],
    'val_auc':        checkpoint['val_auc'],
    'val_bal_acc':    checkpoint['val_bal_acc'],
    'val_f1':         checkpoint['val_f1'],
    'test_auc':       test_m['auc'],
    'test_bal_acc':   test_m['bal_acc'],
    'test_f1':        test_m['f1'],
    'n_test':         len(test_df),
    'n_tnbc_test':    int((test_df['label']==1).sum()),
    'n_nontnbc_test': int((test_df['label']==0).sum())
}
pd.DataFrame([summary]).to_csv(
    f'{RESULT_DIR}/results_summary.csv', index=False
)